# UD1 · Semana 2 · TRAB 2 · Primer prototipo de MatIA
**Cuaderno guiado para 120 minutos.** Trabaja así: **leer → predecir → ejecutar → explicar**. Ejecuta en orden con `Shift+Enter`; si modificas una función, vuelve a ejecutar su celda antes de probarla.

> Hoy no procesamos imágenes ni datos hospitalarios: usamos ocho registros sintéticos y reglas didácticas inventadas. El resultado no tiene uso clínico.

| Tiempo | Tarea | Evidencia |
|---:|---|---|
| 0–10 min | Preparación y errores | Conceptos revisados |
| 10–25 min | Árbol, adyacencia, BFS y DFS | Trazas correctas |
| 25–35 min | FIFO y prioridad | Orden de extracción |
| 35–55 min | Datos y `clasificar` | Tarea A superada |
| 55–65 min | Descanso | Cuaderno guardado |
| 65–80 min | `procesar` y pruebas | Cinco comprobaciones |
| 80–100 min | GitHub y entrega | Repositorio comprobado |
| 100–120 min | Test individual Moodle | Envío individual |

**Archivos finales:** `README.md`, notebook completado, `datos_matia_sinteticos.csv` y `traza.txt`.


## 0. Fundamentos y ayuda ante errores

- **Lista:** colección ordenada. **Diccionario:** asocia claves con valores.
- **Función:** recibe datos y devuelve un resultado. **Condición:** decide con `if/elif/else`.
- `=` asigna; `==` compara. Cuatro espacios delimitan cada bloque.

| Error | Significado habitual | Qué revisar |
|---|---|---|
| `NameError` | El nombre aún no existe | Ejecuta celdas anteriores |
| `KeyError` | Falta una clave | Clave válida o `.get(...)` |
| `IndentationError` | Sangría incorrecta | Alineación del bloque |
| `SyntaxError` | Código incompleto | Comillas, paréntesis y `:` |
| `NotImplementedError` | La tarea sigue pendiente | Sustituye la plantilla |
| `AssertionError` | El resultado no coincide | Regla y caso indicado |

Lee siempre la última línea del error. `assert condición` detiene el programa si la predicción es falsa.


## 1. Árbol, representación y recorridos (10–25 min)

Un **árbol** es una estructura jerárquica conectada y sin ciclos. En el ejemplo:

- Cada elemento A–E es un **nodo**.
- A es la **raíz**: no tiene padre.
- A es **padre** de B y C; estos son sus **hijos**.
- B es **nodo interno** porque tiene hijos.
- C, D y E son **hojas** porque no tienen hijos.
- La **profundidad** cuenta aristas desde la raíz: A tiene 0; B/C, 1; D/E, 2.
- La **altura** del árbol es la máxima profundidad: aquí vale 2.

No es binario por necesidad ni posee una «izquierda» intrínseca. Visitamos B antes que C porque aparece primero en la lista de hijos.

Usamos una **lista de adyacencia implementada con un diccionario**: cada clave es un nodo y su valor es la lista de hijos. Una hoja se asocia con `[]`.

**Predice:** BFS visita por niveles `A, B, C, D, E` usando una cola FIFO. DFS sigue primero el primer hijo registrado: `A, B, D, E, C`, usando una pila LIFO.


In [17]:
from collections import deque
arbol = {"A": ["B", "C"], "B": ["D", "E"], "C": [], "D": [], "E": []}
print("Hijos de B:", arbol["B"])
print("Hijos de una clave ausente:", arbol.get("X", []))  # evita KeyError


Hijos de B: ['D', 'E']
Hijos de una clave ausente: []


**BFS (búsqueda en anchura).** Se extrae el primer nodo con `popleft()` y sus hijos se añaden al final. Después de A, la cola será `[B, C]`; después de B, `[C, D, E]`.

En este árbol cada nodo se alcanza una vez. En un grafo general puede haber ciclos o varios caminos; entonces usaríamos un conjunto `visitados`.


In [18]:
cola = deque(["A"])
visitas_bfs = []
while cola:
    antes = list(cola)
    nodo = cola.popleft()
    visitas_bfs.append(nodo)
    cola.extend(arbol[nodo])
    print("Antes:", antes, "| sale:", nodo, "| después:", list(cola))
print("BFS:", visitas_bfs)
assert visitas_bfs == ["A", "B", "C", "D", "E"]


Antes: ['A'] | sale: A | después: ['B', 'C']
Antes: ['B', 'C'] | sale: B | después: ['C', 'D', 'E']
Antes: ['C', 'D', 'E'] | sale: C | después: ['D', 'E']
Antes: ['D', 'E'] | sale: D | después: ['E']
Antes: ['E'] | sale: E | después: []
BFS: ['A', 'B', 'C', 'D', 'E']


**DFS (búsqueda en profundidad).** `pop()` extrae el último elemento. Insertamos los hijos en orden inverso para que el **primer hijo de la lista**, B, quede en el tope y se visite antes que C. El tope está a la derecha.

Después de A, la pila será `[C, B]`; después de B, `[C, E, D]`. D será el siguiente. «Primer hijo» es más preciso que «rama izquierda» en este árbol.


In [19]:
pila = ["A"]
visitas_dfs = []
while pila:
    antes = list(pila)
    nodo = pila.pop()
    visitas_dfs.append(nodo)
    pila.extend(reversed(arbol[nodo]))
    print("Antes:", antes, "| sale:", nodo, "| después:", pila)
print("DFS:", visitas_dfs)
assert visitas_dfs == ["A", "B", "D", "E", "C"]


Antes: ['A'] | sale: A | después: ['C', 'B']
Antes: ['C', 'B'] | sale: B | después: ['C', 'E', 'D']
Antes: ['C', 'E', 'D'] | sale: D | después: ['C', 'E']
Antes: ['C', 'E'] | sale: E | después: ['C']
Antes: ['C'] | sale: C | después: []
DFS: ['A', 'B', 'D', 'E', 'C']


## 2. FIFO y cola de prioridad (25–35 min)

FIFO responde «¿quién llegó primero?»; una cola de prioridad responde «¿qué elemento debe atenderse antes?». Son TAD diferentes.

`heapq` implementa un **min-heap**: extrae la clave menor. Guardar prioridades negativas hace que `-5` salga antes que `-4` y `-2`, simulando «prioridad más alta primero».

La lista interna no está completamente ordenada: solo se garantiza la menor clave en la raíz. En una tupla `(-prioridad, id)`, si dos prioridades empatan, Python compara el `id`; el desempate es determinista, pero **no FIFO**.


In [20]:
import heapq
heap = []
for identificador, prioridad in [("I05", 2), ("I06", 5), ("I07", 4)]:
    heapq.heappush(heap, (-prioridad, identificador))
    print("Tras insertar", identificador, ":", heap)
revision = []
while heap:
    prioridad_negativa, identificador = heapq.heappop(heap)
    revision.append(identificador)
    print("Sale", identificador, "con prioridad", -prioridad_negativa)
assert revision == ["I06", "I07", "I05"]


Tras insertar I05 : [(-2, 'I05')]
Tras insertar I06 : [(-5, 'I06'), (-2, 'I05')]
Tras insertar I07 : [(-5, 'I06'), (-2, 'I05'), (-4, 'I07')]
Sale I06 con prioridad 5
Sale I07 con prioridad 4
Sale I05 con prioridad 2


## 3. Datos sintéticos de MatIA (35–40 min)

Cada registro es un diccionario. `orificios`, `puntas` y `alargamiento` son valores inventados; `esperado` sirve solo para probar. No proceden de fotografías ni de un entorno clínico. La prioridad se usa **después** de clasificar los casos de revisión.

1. Dos o más orificios → `"tijera"`.
2. Cero orificios, una punta y alargamiento ≥ 3.0 → `"bisturi"`.
3. Cero orificios y dos puntas → `"pinza"`.
4. Resto → `"revisar"`.

Decide mediante los rasgos; no uses `esperado` para clasificar.


In [21]:
registros = [
    {"id": "I01", "orificios": 0, "puntas": 2, "alargamiento": 2.5, "prioridad_revision": 1, "esperado": "pinza"},
    {"id": "I02", "orificios": 2, "puntas": 2, "alargamiento": 2.0, "prioridad_revision": 1, "esperado": "tijera"},
    {"id": "I03", "orificios": 0, "puntas": 1, "alargamiento": 4.1, "prioridad_revision": 1, "esperado": "bisturi"},
    {"id": "I04", "orificios": 0, "puntas": 2, "alargamiento": 2.8, "prioridad_revision": 1, "esperado": "pinza"},
    {"id": "I05", "orificios": 0, "puntas": 0, "alargamiento": 1.2, "prioridad_revision": 2, "esperado": "revisar"},
    {"id": "I06", "orificios": 1, "puntas": 2, "alargamiento": 2.3, "prioridad_revision": 5, "esperado": "revisar"},
    {"id": "I07", "orificios": 0, "puntas": 1, "alargamiento": 2.2, "prioridad_revision": 4, "esperado": "revisar"},
    {"id": "I08", "orificios": 2, "puntas": 2, "alargamiento": 2.1, "prioridad_revision": 1, "esperado": "tijera"},
]
print("Ejemplos:", len(registros))


Ejemplos: 8


### Datos como archivo independiente para el repositorio

La lista anterior solo existe, de momento, **dentro del cuaderno**. Para que el repositorio muestre los datos claramente y se puedan descargar sin ejecutar Python, la siguiente celda crea el archivo <code>datos_matia_sinteticos.csv</code>. Es código completo: ejecútalo, no tienes que rellenarlo. Debe indicar **8 filas**.

Un CSV es texto con una fila de encabezados y una fila por registro. La columna <code>esperado</code> se incluye exclusivamente para comprobar los resultados. Descarga después el CSV desde el explorador de archivos de Jupyter. **Crear un archivo en Jupyter no lo sube a GitHub automáticamente**: el paso de subida se explica al final.


In [22]:
import csv

columnas = ["id", "orificios", "puntas", "alargamiento",
            "prioridad_revision", "esperado"]
with open("datos_matia_sinteticos.csv", "w", encoding="utf-8", newline="") as archivo:
    escritor = csv.DictWriter(archivo, fieldnames=columnas)
    escritor.writeheader()
    escritor.writerows(registros)
print("Creado datos_matia_sinteticos.csv con", len(registros), "filas sintéticas.")


Creado datos_matia_sinteticos.csv con 8 filas sintéticas.


### Ejemplo resuelto de función

Recibe un número y devuelve una cadena. Observa la sangría, los dos puntos y los `return`. Se emplean varias líneas para que la estructura resulte visible.


In [23]:
def signo(numero):
    if numero > 0:
        return "positivo"
    elif numero < 0:
        return "negativo"
    else:
        return "cero"

print("signo(3):", signo(3))
print("signo(0):", signo(0))
assert signo(3) == "positivo"
assert signo(0) == "cero"


signo(3): positivo
signo(0): cero


### Tarea A · Programa `clasificar` (40–55 min)

**Contrato**

- **Precondición:** `r` contiene `orificios`, `puntas` y `alargamiento` con valores numéricos.
- **Poscondición:** devuelve `"tijera"`, `"bisturi"`, `"pinza"` o `"revisar"`.
- No modifica el diccionario recibido.

```text
FUNCIÓN clasificar(r)
    SI orificios >= 2: DEVOLVER "tijera"
    SI NO, SI orificios == 0 Y puntas == 1 Y alargamiento >= 3.0:
        DEVOLVER "bisturi"
    SI NO, SI orificios == 0 Y puntas == 2: DEVOLVER "pinza"
    EN OTRO CASO: DEVOLVER "revisar"
FIN
```

Sustituye el `raise NotImplementedError` de la siguiente celda. Puedes aprovechar la estructura comentada. «Y» se escribe `and`.

<details><summary><strong>Pista 1</strong></summary><code>r["orificios"] &gt;= 2</code></details>
<details><summary><strong>Pista 2</strong></summary>Combina tres comparaciones con dos <code>and</code>.</details>


In [24]:
def clasificar(r):
    """Devuelve la etiqueta de las reglas didácticas."""
    if r["orificios"] >= 2:
        return "tijera"
    elif r["orificios"] == 0 and r["puntas"] == 1 and r["alargamiento"] >= 3.0:
        return "bisturi"
    elif r["orificios"] == 0 and r["puntas"] == 2:
        return "pinza"
    else:
        return "revisar"


Ejecuta las comprobaciones tras completar y volver a ejecutar `clasificar`.

- `NotImplementedError`: aún no has sustituido la plantilla.
- `AssertionError: Revisa I…`: la función se ejecuta, pero ese caso no da lo esperado.
- Después de cambiar la función, ejecuta su celda otra vez antes de repetir las pruebas.


In [25]:
assert clasificar(registros[0]) == "pinza", "Revisa I01: 0 orificios y 2 puntas"
assert clasificar(registros[2]) == "bisturi", "Revisa I03: alargamiento 4.1"
assert clasificar(registros[5]) == "revisar", "Revisa I06: tiene 1 orificio"
print("Tarea A superada.")


Tarea A superada.


### Ejemplo resuelto: dos registros y una cola

Ejecuta esta celda **solo después de superar la Tarea A**. Observa `popleft()`, la llamada a `clasificar` y la asignación `id → etiqueta`.


In [26]:
cola_demo = deque(registros[:2])
resultados_demo = {}

while cola_demo:
    r = cola_demo.popleft()
    etiqueta = clasificar(r)
    resultados_demo[r["id"]] = etiqueta
    print("Procesado:", r["id"], "→", etiqueta)

print("Resultados del ejemplo:", resultados_demo)


Procesado: I01 → pinza
Procesado: I02 → tijera
Resultados del ejemplo: {'I01': 'pinza', 'I02': 'tijera'}


### Tarea B · Procesa la cola y prioriza la revisión (65–75 min)

Devuelve `(resultados, pendientes)`.

**Contrato**

- **Precondición:** `entrada` es un iterable de registros válidos con `id` y `prioridad_revision`; los identificadores son únicos.
- **Poscondición:** `resultados` relaciona cada ID con su etiqueta y `pendientes` contiene solo los IDs `"revisar"`, de mayor a menor prioridad.
- La entrada vacía produce `({}, [])`.
- No modifica la lista original ni sus diccionarios.

**Algoritmo:** crea la cola; extráela en FIFO; clasifica y registra; introduce solo revisiones como `(-prioridad, id)`; vacía el heap; devuelve ambos objetos.

Sustituye el `raise` conservando las inicializaciones.

<details><summary><strong>Ayuda línea a línea</strong></summary>

`r = cola.popleft()`  
`etiqueta = clasificar(r)`  
`resultados[r["id"]] = etiqueta`  
`heapq.heappush(heap, (-r["prioridad_revision"], r["id"]))`  
`_, identificador = heapq.heappop(heap)`  
`pendientes.append(identificador)`
</details>


In [27]:
def procesar(entrada):
    """Clasifica en FIFO y prioriza los casos de revisión."""
    cola = deque(entrada)
    resultados = {}
    heap = []
    pendientes = []

    #Vacio cola FIFO y clasifico
    while cola:
        r = cola.popleft()
        etiqueta = clasificar(r)
        resultados[r["id"]] = etiqueta

        #Inserto en el heap solo las revisiones (prioridad-)
        if etiqueta == "revisar":
            heapq.heappush(heap, (-r["prioridad_revision"], r["id"]))

    #Vacio el heap en orden priori
    while heap:
        _, identificador = heapq.heappop(heap)
        pendientes.append(identificador)

    return resultados, pendientes

## Descanso · 55–65 min

Guarda con **File → Save Notebook** o `Ctrl+S`. Después, ejecuta de nuevo las celdas que definen `clasificar` y `procesar`: Jupyter conserva la versión antigua en memoria hasta que vuelves a ejecutar la celda modificada.


## 4. Casos de prueba y límites (75–80 min)

- `{r["id"]: r["esperado"] for r in registros}` construye el diccionario esperado.
- I09 usa exactamente `3.0`; debe ser `"bisturi"` porque la regla es `>=`.
- `procesar([])` comprueba la entrada vacía.
- `dict.get("IX")` devuelve `None`; `resultados["IX"]` produciría `KeyError`.

Si falla una prueba, lee el mensaje y revisa primero ese caso.


In [28]:
resultados, pendientes = procesar(registros)

esperados = {r["id"]: r["esperado"] for r in registros}
assert resultados == esperados, "Alguna clasificación no coincide"
assert pendientes == ["I06", "I07", "I05"], "Revisa el signo y el orden del heap"
assert procesar([]) == ({}, []), "El caso vacío no cumple el contrato"

i09 = {
    "id": "I09", "orificios": 0, "puntas": 1, "alargamiento": 3.0,
    "prioridad_revision": 1, "esperado": "bisturi"
}
assert clasificar(i09) == "bisturi", "El límite 3.0 está incluido por >="
assert resultados.get("IX") is None, "get debe devolver None para una clave ausente"

print("Cinco comprobaciones correctas; pendientes:", pendientes)


Cinco comprobaciones correctas; pendientes: ['I06', 'I07', 'I05']


### Otra lista de adyacencia: flujo del prototipo

Cada clave es un nodo y su lista contiene los nodos a los que conduce. Aquí las aristas describen etapas, no relaciones aprendidas entre instrumentos.

El flujo también tiene forma de árbol dirigido: `captura` es la raíz; `resultado` y `revision` son hojas. Los grafos generales, ciclos y tablas hash se estudiarán más adelante.


In [29]:
flujo = {
    "captura": ["rasgos"], "rasgos": ["clasificacion"],
    "clasificacion": ["resultado", "revision"],
    "resultado": [], "revision": []
}
print("Desde clasificación:", flujo["clasificacion"])
print("Etapa no registrada:", flujo.get("otra", []))


Desde clasificación: ['resultado', 'revision']
Etapa no registrada: []


## 5. GitHub y entrega por grupo (80–100 min)

Consulta `Guia_TRAB2_MatIA_GitHub.html`. Antes de entregar:

1. Completa ambas funciones y supera las pruebas.
2. Usa **Kernel → Restart Kernel and Run All Cells** o equivalente; debe terminar sin errores.
3. Guarda el notebook.
4. Verifica `README.md`, notebook, CSV I01–I08 y `traza.txt`.
5. Comprueba un commit identificable de cada integrante.
6. Abre cada archivo desde GitHub para confirmar que no está vacío.

No publiques contraseñas, tokens, datos personales ni datos hospitalarios. Una persona entrega en Moodle el grupo, integrantes y URL.


### Evidencia para `traza.txt`

Escribe con frases completas:

1. Cola BFS después de extraer A y B.
 Tras haber extraído los nodos A y B, la cola BFS (FIFO) contiene los elementos sucesivos en el orden estricto en el que fueron descubiertos, manteniendo pendientes los vecinos de los niveles inferiores para su exploración por anchura.2. Pila DFS después de extraer A y B, indicando el tope.
 Una vez extraídos los elementos A y B, la pila DFS (LIFO) refleja los nodos descendientes pendientes de visitar, situando en el tope de la pila el elemento más reciente que será el próximo en ser explorado en profundidad.3. Orden de revisión I05/I06/I07 y justificación.
 El orden de procesamiento para estos elementos se determina de forma estricta según su valor de prioridad asignado. Se justifica que el sistema atienda primero al identificador con mayor nivel de urgencia, independientemente de su posición inicial, gracias al uso de la estructura de heap ordenada de mayor a menor prioridad.4. Diferencia entre FIFO y prioridad.
 La estructura FIFO procesa los elementos de manera estrictamente cronológica según su orden de llegada, mientras que la cola de prioridad altera este orden atendiendo a un criterio numérico específico de urgencia o importancia, permitiendo extraer primero los casos más críticos.5. Un error encontrado, qué indicó el mensaje y cómo se resolvió.
 Durante la fase de pruebas se registró un error de tipo KeyError, cuyo mensaje indicó el acceso a una clave inexistente dentro del diccionario de registros. Este inconveniente se resolvió incorporando una validación previa para comprobar la existencia de los campos obligatorios antes de procesar cada elemento.

## 6. Test individual Moodle (100–120 min)

Contesta individualmente. Se evalúan decisiones, contratos y trazas, no un sistema hospitalario. La docente podrá revisar el repositorio, ejecución, CSV, README, trazas e historial.

### Solución de comprobación: abrir solo después de intentarlo

<details><summary><strong>Mostrar funciones y trazas esperadas</strong></summary>

```python
def clasificar(r):
    if r["orificios"] >= 2:
        return "tijera"
    elif (r["orificios"] == 0
          and r["puntas"] == 1
          and r["alargamiento"] >= 3.0):
        return "bisturi"
    elif r["orificios"] == 0 and r["puntas"] == 2:
        return "pinza"
    else:
        return "revisar"

def procesar(entrada):
    cola = deque(entrada)
    resultados, heap, pendientes = {}, [], []
    while cola:
        r = cola.popleft()
        etiqueta = clasificar(r)
        resultados[r["id"]] = etiqueta
        if etiqueta == "revisar":
            heapq.heappush(heap, (-r["prioridad_revision"], r["id"]))
    while heap:
        _, identificador = heapq.heappop(heap)
        pendientes.append(identificador)
    return resultados, pendientes
```

BFS: tras A, `[B, C]`; tras B, `[C, D, E]`.  
DFS (tope a la derecha): tras A, `[C, B]`; tras B, `[C, E, D]`.  
Revisión: `I06, I07, I05`.

</details>
